# Train 3-branch model (1x3D ResNeXt + 2x2D MaxViT + CrossGate) + optional information-theory survey

Canonical training notebook for the repo's recommended main model: one 3D ResNeXt branch over the OCT volume
(96^3, resized on the fly from the raw 200^3 storage) plus two 2D MaxViT-Tiny branches over the en-face views
(`slab_mip` + `aip_full`, projected from the raw 200^3 volume), fused by CrossGate and classified by a linear
head. Architecture lives in `scripts/final_model.py`; the resumable trainer, metrics, calibration, reporting
and X-AI live in `scripts/final_training.py`; the optional analysis lives in `scripts/information_theory.py`.

- `N_2D=2` gives the 3-branch model (default); `N_2D=1` gives the 1x3D + slab_mip ablation.
- `RUN_XAI=True` explains the best model (Grad-CAM 3D/2D, occlusion, integrated gradients, fusion attention,
  branch drop) and logs `xai/fusion_table` + `xai/*` to W&B.
- `RUN_INFO=True` additionally runs the information-theory survey (linear probes, MINE/NWJ, MIC, surrogate
  data testing, joint/conditional MI, information plane) and logs `info_theory.*` + figures. Per-epoch
  validation embeddings are captured only when this flag is on.

Storage resolution is config-driven (`STORE_RES`, default raw 200^3); downloads always use authenticated
`HF_TOKEN` with `allow_patterns` for the declared splits. Logging follows `docs/notebook-conventions.md`: train
metrics every optimizer step, val + test metrics every epoch, calibrated `train/val/test` + bootstrap CI +
`report/split_table` in W&B, and every artifact local + Drive.

`TB3_SMOKE=1` runs tiny synthetic CPU data with offline W&B. Real runs need a GPU plus `WANDB_API_KEY` and `HF_TOKEN`.

In [ ]:
import os, sys, subprocess
from pathlib import Path
SMOKE = os.environ.get('TB3_SMOKE', '0') == '1'
if not SMOKE and 'google.colab' in sys.modules:
    repo = Path('/content/glaucoma-thesis')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'wandb', 'scipy', 'matplotlib', 'huggingface_hub', 'python-dotenv', 'hf-transfer'], check=True)
    os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
sys.path.insert(0, str(Path.cwd()))
import json, random, tempfile, time
import numpy as np
import torch
import torch.nn.functional as F
import wandb
import matplotlib
matplotlib.use('Agg')
from scripts import final_model as fm, final_training as ft, information_theory as it
ft.load_env_file()
DEVICE = torch.device('cpu' if SMOKE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
if SMOKE:
    torch.set_num_threads(1)


## Configuration
`STORE_RES` is the on-disk storage resolution (200/128/96/...); `RES3D` is the 3D model input (resized on the
fly) and `RES2D` the 2D view cache size. Defaults store raw 200^3 (`harvard-oct-glaucoma-200`) but train the 3D
branch at 96^3 (`RES3D=96`), with 2D views projected from the raw volume; change `STORE_RES`/`RES3D` and
`HF_DATA_REPO` together to switch storage. `RUN_XAI` and `RUN_INFO` gate the optional stages in this same
notebook (no forks). Change `RUN_GROUP` for every new config; `RESUME=True` requires an identical training
config, and `WARM_START_WEIGHTS` can transfer a 200^3-trained model (fully convolutional encoder).

In [ ]:
RUN_GROUP = 'crossgate_3branch_s42'
RUN_TARGET = 'raw_s42'
RESUME = False
WARM_START_WEIGHTS = ''
CHECKPOINT_EVERY_STEPS = 10
DATASETS = ['raw']
SEEDS = [42]
N_2D = 1 if SMOKE else 2
STORE_RES = 8 if SMOKE else 200
RES3D = 8 if SMOKE else 96
RES2D = 8 if SMOKE else 224
D_LATENT, ENC2D = 256, 'maxvit_tiny_rw_224'
ENC3D_FEATURES = (32, 64, 128, 192)
EPOCHS, BS, GRAD_ACCUM = (1, 2, 2) if SMOKE else (10, 2, 8)
LR, WD, PATIENCE = 1e-4, 1e-4, 4
RUN_XAI = True
RUN_INFO = True
IT_MINE_STEPS = 40 if SMOKE else 600
IT_MINE_HIDDEN = 32 if SMOKE else 64
IT_PROBE_STEPS = 60 if SMOKE else 400
IT_SURROGATES = 3 if SMOKE else 100
IT_PCA_DIM = 6 if SMOKE else 32
IT_INPUT_RES = 4 if SMOKE else 16
IT_PLANE_FOLDS = 2 if SMOKE else 3
IT_MIC_PCS = 4 if SMOKE else 8
HF_DATA_REPO = os.environ.get('HF_DATA_REPO', 'tqhuyen/harvard-oct-glaucoma-200')
SPLITS = ('Training', 'Validation', 'Test')
HF_DATA_PATTERNS = [f'{split}_{kind}.npy' for split in SPLITS for kind in ('volumes', 'labels')]
SMOKE_ROOT = Path(tempfile.mkdtemp(prefix='tb3_smoke_')) if SMOKE else None
DATA_ROOT = SMOKE_ROOT / 'data' if SMOKE else Path('/content/final_data')
LOCAL_ROOT = SMOKE_ROOT / 'runs' if SMOKE else Path('outputs/crossgate_3branch') / RUN_GROUP
DRIVE_MOUNT = Path(os.environ.get('DRIVE_MOUNT', '/content/drive'))
DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/MasterBKDN/Thesis'))
DRIVE_DIR = DRIVE_ROOT / 'crossgate_3branch' / RUN_GROUP
selected = [(ds, seed, f'{ds}_s{seed}') for ds in DATASETS for seed in SEEDS if not RUN_TARGET or RUN_TARGET == f'{ds}_s{seed}']
if not selected:
    raise ValueError('RUN_TARGET does not match DATASETS/SEEDS')
if N_2D not in (1, 2):
    raise ValueError('N_2D must be 1 (ablation) or 2 (3-branch model)')
if RES3D > STORE_RES:
    raise ValueError('RES3D cannot exceed the stored resolution')
if WARM_START_WEIGHTS and (RESUME or len(selected) != 1):
    raise ValueError('Warm-start requires RESUME=False and one explicitly selected RUN_TARGET')
if not SMOKE:
    if not os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not DRIVE_ROOT.resolve().is_relative_to(DRIVE_MOUNT.resolve()):
        raise RuntimeError('Drive must be a verified mounted filesystem, not a local directory')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
if WARM_START_WEIGHTS and not Path(WARM_START_WEIGHTS).is_file():
    raise FileNotFoundError(WARM_START_WEIGHTS)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
STORAGE = ft.Artifacts(LOCAL_ROOT, None if SMOKE else DRIVE_DIR, smoke=SMOKE)
DATA_STORAGE = ft.Artifacts(DATA_ROOT, None if SMOKE else DRIVE_DIR / 'data', smoke=SMOKE)
print('Device:', DEVICE, '| branches: 3D +', N_2D, 'x 2D | storage', STORE_RES, '| res3d', RES3D, '| res2d', RES2D, '| xai', RUN_XAI, '| info', RUN_INFO)


## Data
This preset stores and reads raw **200^3** volumes (`tqhuyen/harvard-oct-glaucoma-200`); the 3D branch trains on
**96^3** tensors (trilinear resize on the fly, no downsampled arrays persisted) while the 2D views are
projected from the **raw 200^3** volume and cached at `RES2D=224`. Storage resolution stays config-driven
(`STORE_RES`: 200/128/96/...; change it together with `HF_DATA_REPO`). Real runs download **only the declared
splits/files** with `HF_TOKEN` (`allow_patterns`), required for maximum authenticated throughput. Existing
caches are reused; expensive processed data (denoise/views/augmentation) should be considered for a versioned
HF upload so later runs skip recompute. Smoke builds tiny synthetic arrays and never downloads.

In [ ]:
if SMOKE:
    rng = np.random.default_rng(0)
    for split, n in zip(SPLITS, (10, 6, 6)):
        np.save(DATA_ROOT / f'{split}_volumes.npy', rng.integers(0, 255, (n, 1, STORE_RES, STORE_RES, STORE_RES), dtype=np.uint8))
        np.save(DATA_ROOT / f'{split}_labels.npy', np.arange(n, dtype=np.int64) % 2)
else:
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not token:
        raise RuntimeError('Authenticated HF download requires HF_TOKEN in .env/environment or Colab Secrets')
    if not all((DATA_ROOT / name).is_file() for name in HF_DATA_PATTERNS):
        snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=str(DATA_ROOT), token=token, allow_patterns=HF_DATA_PATTERNS)
def make_datasets(ds, seed):
    datasets = [ft.FinalDataset(DATA_ROOT / f'{s}_volumes.npy', DATA_ROOT / f'{s}_labels.npy', res3d=RES3D, res2d=RES2D, seed=seed, train=s == 'Training') for s in SPLITS]
    if not SMOKE and any(tuple(d.volumes.shape[-3:]) != (STORE_RES, STORE_RES, STORE_RES) for d in datasets):
        raise ValueError(f'Real training requires STORE_RES-cubed storage, got {STORE_RES}')
    if ds != 'raw':
        raise ValueError('This notebook trains on raw storage; use the final crossgate notebook for denoised runs')
    for split, dataset in zip(SPLITS, datasets):
        print(f'[input] {split}: {dataset.source} | res3d={RES3D} | views cached at {RES2D}px')
    for split in SPLITS:
        for path in DATA_ROOT.glob(f'{split}_volumes_*{RES2D}*'):
            if '.partial.' not in path.name:
                DATA_STORAGE.sync(path)
    return datasets


## Train, evaluate, persist and explain
Metrics cadence: every optimizer step logs the full metric set on the current accumulation window under
`train/*` (plus `train/loss`, running `train/acc`, `train/lr`, `progress/step`); validation and test log the
full set every epoch under `val/*` and `test/*`. After training, `calibrated_report` adds calibrated
`train/val/test` metrics plus bootstrap CI and `log_report` writes the `report/split_table` W&B table.
`RUN_XAI=True` then explains the best model and logs `xai/fusion_table` + `xai/*`. When `RUN_INFO=True`, the
per-epoch `evaluate` also captures embeddings for the information plane, and the analysis cell below finishes
the W&B run. Best state is selected by validation AUC, calibrated on validation, then evaluated once on the
held-out test set.

In [ ]:
RESULTS = {}
ACTIVE_MODEL = ACTIVE_TRAINER = WANDB_RUN = None
LAST = {}
for ds, seed, tag in selected:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va, te = make_datasets(ds, seed)
    ytr = tr.labels
    if set(np.unique(ytr)) != {0, 1}:
        raise ValueError('Training requires both binary classes')
    weights = [len(ytr) / (2 * int((ytr == c).sum())) for c in (0, 1)]
    config = dict(dataset=ds, seed=seed, epochs=EPOCHS, batch_size=BS, grad_accum=GRAD_ACCUM, lr=LR, weight_decay=WD, patience=PATIENCE, checkpoint_steps=CHECKPOINT_EVERY_STEPS, class_weights=weights, res3d=RES3D, res2d=RES2D, store_res=STORE_RES, n2d=N_2D, latent=D_LATENT, enc2d=ENC2D, enc3d_features=list(ENC3D_FEATURES), smoke=SMOKE, torch_version=str(torch.__version__), device=str(DEVICE), data=[ft.data_identity(p) for d in (tr, va, te) for p in (d.source, d.label_path)])
    artifacts = ft.Artifacts(LOCAL_ROOT / tag, None if SMOKE else DRIVE_DIR / tag, smoke=SMOKE)
    status = ft.run_status(artifacts, config, resume=RESUME)
    print(tag, status['status'])
    LAST = {'tag': tag, 'artifacts': artifacts, 'config': config}
    if status['status'] == 'complete':
        RESULTS[tag] = {'res': status['result']}
        continue
    tag_resume = status['status'] == 'resume'
    tag_warm_start = status['warm_start'] if status['status'] == 'initialized' else (WARM_START_WEIGHTS if tag == RUN_TARGET else '')
    if tag_warm_start and not Path(tag_warm_start).is_file():
        raise FileNotFoundError('Uncommitted warm-start requires its original weights: ' + tag_warm_start)
    WANDB_RUN = ft.init_wandb('crossgate_3branch_' + tag, config, artifacts, resume=status['status'] != 'new', smoke=SMOKE, warm_start=tag_warm_start)
    stopped, started = False, time.time()
    epoch_embeddings = {'z': [], 'e3d': [], 'e2d': [], 'x': [], 'y': []}
    try:
        ACTIVE_MODEL = (ft.SmokeModel() if SMOKE else fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=not (tag_resume or bool(tag_warm_start)))).to(DEVICE)
        if RUN_INFO:
            def evaluate(model):
                capture = it.collect_embeddings(model, va, BS, input_res=IT_INPUT_RES)
                epoch_embeddings['z'].append(capture['z'])
                epoch_embeddings['e3d'].append(capture['e3d'])
                epoch_embeddings['e2d'].append(capture['e2d'])
                epoch_embeddings['y'].append(capture['y'])
                if not epoch_embeddings['x']:
                    epoch_embeddings['x'].append(capture['x'])
                return {**fm.full_metrics(capture['probs'], capture['y']), 'loss': float(F.cross_entropy(torch.tensor(capture['logits']), torch.tensor(capture['y'])))}
        else:
            def evaluate(model):
                p, y, logits = ft.predict(model, va, BS)
                return {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        def evaluate_test(model):
            p, y, logits = ft.predict(model, te, BS)
            return {**fm.full_metrics(p, y), 'loss': float(F.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        ACTIVE_TRAINER = ft.Trainer(ACTIVE_MODEL, tr, config, artifacts, WANDB_RUN, resume=tag_resume, warm_start=tag_warm_start)
        stopped = not ACTIVE_TRAINER.fit(evaluate, test_evaluate=evaluate_test)
        if not stopped:
            ACTIVE_MODEL.load_state_dict(ACTIVE_TRAINER.best_state)
            res, probs, labels = ft.calibrated_report(ACTIVE_MODEL, va, te, BS, smoke=SMOKE, train=tr)
            res.update(tag=tag, seed=seed, hist=ACTIVE_TRAINER.history, minutes=round((time.time() - started) / 60, 2))
            ft.log_report(WANDB_RUN, res)
            params = sum(p.numel() for p in ACTIVE_MODEL.parameters())
            WANDB_RUN.summary.update({'threshold': res['threshold'], 'temperature': res['temperature'], 'params': params, 'minutes': res['minutes']})
            weights_path = artifacts.save(ft.cpu_state(ACTIVE_MODEL), 'best_weights.pt')
            ft.save_report(res, probs, labels, artifacts, WANDB_RUN)
            RESULTS[tag] = dict(res=res, weights_path=str(weights_path), params=params, test_probs=probs.tolist(), test_labels=labels.tolist())
            if RUN_XAI:
                ft.save_xai(ACTIVE_MODEL, va, artifacts, WANDB_RUN, smoke=SMOKE)
    except BaseException:
        WANDB_RUN.finish(exit_code=1)
        raise
    if stopped:
        WANDB_RUN.summary['stopped_safely'] = True
        WANDB_RUN.finish(exit_code=0)
        print('Stopped safely. Set RESUME=True and RUN_TARGET to', tag)
        break
    ft.complete_run(artifacts, config, WANDB_RUN.id)
    ACTIVE_MODEL.cpu()
    ACTIVE_TRAINER = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Completed:', list(RESULTS))


## Information-theory survey (optional, `RUN_INFO`)
Fit light probes on training embeddings and score on test; estimate MI on validation and test with MINE (DV)
and NWJ; approximate MIC over PCA components; run surrogate label permutations for a null distribution and
p-value; compute joint/conditional MI between ResNeXt and view embeddings; and build the information plane
I(X;Z) vs I(Z;Y) from the per-epoch embeddings captured during training. Everything is saved to
`info_theory.json` / `info_theory.pt` and logged to W&B; figures are synced to Drive. When `RUN_INFO=False`
this cell only closes the W&B run.

In [ ]:
info_results = {}
if RUN_INFO and RESULTS:
    tag = LAST['tag']
    analysis_model = (ft.SmokeModel() if SMOKE else fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc3d_features=ENC3D_FEATURES, enc2d_pretrained=False)).to(DEVICE)
    weights = LOCAL_ROOT / tag / 'best_weights.pt'
    if not weights.is_file():
        raise FileNotFoundError(weights)
    analysis_model.load_state_dict(torch.load(weights, map_location='cpu'))
    print('Analysis model:', weights)
    tr_e = it.collect_embeddings(analysis_model, tr, BS, input_res=IT_INPUT_RES)
    va_e = it.collect_embeddings(analysis_model, va, BS, input_res=IT_INPUT_RES)
    te_e = it.collect_embeddings(analysis_model, te, BS, input_res=IT_INPUT_RES)
    def branch(capture, key):
        if key == 'view0':
            return capture['e2d'][:, 0]
        if key == 'view1':
            return capture['e2d'][:, 1]
        return capture[key]
    branches = {'z_fused': 'z', 'resnext3d': 'e3d', 'view0': 'view0'}
    if N_2D > 1:
        branches['view1'] = 'view1'
    probe_rows = []
    for name, key in branches.items():
        for kind in ('logistic', 'mlp'):
            result = it.linear_probe(branch(tr_e, key), tr_e['y'], branch(te_e, key), te_e['y'], kind=kind, steps=IT_PROBE_STEPS, seed=SEEDS[0], device=str(DEVICE))
            probe_rows.append({'name': name if kind == 'logistic' else name + '_mlp', 'kind': kind, 'acc': result['acc'], 'auc': result['auc'], 'f1': result['f1']})
    info_results['probes'] = probe_rows
    mi_rows = {}
    for split, capture in (('val', va_e), ('test', te_e)):
        for name, key in branches.items():
            features = branch(capture, key)
            mi_rows[f'{split}/{name}'] = {
                'mine_dv': it.mine_mi(features, capture['y'][:, None], steps=IT_MINE_STEPS, hidden=IT_MINE_HIDDEN, seed=SEEDS[0], device=str(DEVICE))['mi'],
                'mine_nwj': it.mine_mi(features, capture['y'][:, None], bound='nwj', steps=IT_MINE_STEPS, hidden=IT_MINE_HIDDEN, seed=SEEDS[0], device=str(DEVICE))['mi'],
                'max_mic': it.max_mic_features(features, capture['y'], pcs=IT_MIC_PCS)['max_mic'],
            }
    info_results['mi'] = mi_rows
    surrogate = {}
    for name in ('z_fused', 'resnext3d'):
        surrogate[name] = it.surrogate_test(lambda a, b: it.mine_mi(a, b, steps=IT_MINE_STEPS, hidden=IT_MINE_HIDDEN, seed=SEEDS[0], device=str(DEVICE))['mi'], branch(va_e, branches[name]), va_e['y'], n=IT_SURROGATES, seed=SEEDS[0])
    projected = it.PCA(IT_MIC_PCS).fit_transform(branch(va_e, branches['z_fused']))
    surrogate['z_fused_mic'] = it.surrogate_test(lambda a, b: it.mic_approx(a[:, 0], b)['mic'], projected, va_e['y'], n=IT_SURROGATES, seed=SEEDS[0])
    info_results['surrogate'] = surrogate
    branch_dim = max(2, min(8, IT_PCA_DIM))
    z3 = it.PCA(branch_dim).fit_transform(branch(va_e, branches['resnext3d']))
    z2 = it.PCA(branch_dim).fit_transform(branch(va_e, branches['view0']))
    info_results['branch_mi'] = {
        'joint_ksg': it.ksg_mi(z3, z2),
        'conditional': it.conditional_mi(z3, va_e['y'][:, None], z2, steps=IT_MINE_STEPS, hidden=IT_MINE_HIDDEN, seed=SEEDS[0], device=str(DEVICE)),
    }
    plane = []
    if epoch_embeddings['z']:
        plane = it.information_plane(epoch_embeddings['x'][0], epoch_embeddings['z'], epoch_embeddings['y'][0], folds=IT_PLANE_FOLDS, pca_dim=IT_PCA_DIM, probe_steps=IT_PROBE_STEPS, mine_steps=IT_MINE_STEPS, seed=SEEDS[0], device=str(DEVICE), hidden=IT_MINE_HIDDEN)
    info_results['plane'] = plane
    info_results['estimator_check'] = it.validate_estimators(n=200 if SMOKE else 800, steps=60 if SMOKE else 500, seed=SEEDS[0], device=str(DEVICE))
    analysis_run = WANDB_RUN if WANDB_RUN is not None else ft.init_wandb('crossgate_3branch_' + tag, LAST['config'], LAST['artifacts'], resume=True, smoke=SMOKE)
    figures = {}
    if plane:
        figures['information_plane'] = it.plot_information_plane(plane, LOCAL_ROOT / 'information_plane.png')
    figures['mi_surrogate'] = it.plot_surrogate(surrogate, LOCAL_ROOT / 'mi_surrogate.png')
    figures['linear_probes'] = it.plot_probes(probe_rows, LOCAL_ROOT / 'linear_probes.png')
    LAST['artifacts'].save(info_results, 'info_theory.pt')
    json_path = LOCAL_ROOT / 'info_theory.json'
    json_path.write_text(json.dumps(info_results, indent=2, default=float))
    LAST['artifacts'].sync(json_path)
    for path in figures.values():
        LAST['artifacts'].sync(Path(path))
        analysis_run.log({f'figures/{Path(path).stem}': wandb.Image(str(path))})
    summary = {f'probes/{row["name"]}_auc': row['auc'] for row in probe_rows}
    summary.update({f'mi/{key}_dv': value['mine_dv'] for key, value in mi_rows.items()})
    summary.update({f'surrogate/{key}_p': value['p_value'] for key, value in surrogate.items()})
    summary.update({'estimator/xor_mi': info_results['estimator_check']['xor_mi']})
    analysis_run.log(summary)
    analysis_run.summary.update(summary)
    analysis_run.finish(exit_code=0)
    ft.complete_run(LAST['artifacts'], LAST['config'], analysis_run.id)
    print('Information-theory artifacts:', sorted(info_results))
else:
    if WANDB_RUN is not None:
        WANDB_RUN.finish(exit_code=0)
    print('Information-theory analysis skipped (RUN_INFO=False or no completed run).')


## Outputs and limitations
- `metrics.json` / `test_predictions.pt` / `curves.png` / `test_report.png`: calibrated `train/val/test` metrics,
  bootstrap CI, learning curves, ROC/PR/calibration/confusion.
- W&B: per-step `train/*`, per-epoch `val/*` + `test/*`, calibrated `train|val|test/*` with CI,
  `report/split_table`, `xai/fusion_table` + `xai/*`, and (when `RUN_INFO`) `probes/*`, `mi/*`, `surrogate/*`.
- `info_theory.json` / `info_theory.pt` + `information_plane.png` / `mi_surrogate.png` / `linear_probes.png`
  when `RUN_INFO=True`.
- Drive: `MasterBKDN/Thesis/crossgate_3branch/<RUN_GROUP>/<tag>/` (plus `data/` caches).
- Trains the 3D branch at 96^3 with raw 200^3 storage (2D views from raw 200^3); switching storage requires
  matching `STORE_RES`/`RES3D`/`HF_DATA_REPO` and a new `RUN_GROUP`.
- Single-seed by default; run 3-5 seeds for mean +/- std. `N_2D=1` is the ablation baseline for the number of
  2D branches. Resume only replays work since the last committed optimizer step.

In [ ]:
rows = ft.publish_summary(STORAGE)
if rows:
    print(rows)
else:
    print('No completed runs; no summary attempted.')
print('Smoke verified only local/offline behavior.' if SMOKE else 'Completed artifacts were synced to the verified Drive mount.')
